# Fine-tune PointPillars on Livox Mid-360S data (Colab)

This notebook assembles the full training pipeline for the Mid-360S fine-tune:

**clone repo -> mount Drive -> unzip data -> build OpenPCDet -> inject our dataset class + configs -> full smoke test -> overfit training -> save checkpoint.**

**Goal:** an *overfit pipeline-proof* run - train on all 48 hand-labeled frames (no val, no eval) and watch the loss fall toward ~0. That proves the data->label->model plumbing is correct end-to-end before investing in a real split and tuning.

**Prerequisites**
- **GPU runtime**: Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.
- Data bundle in your Drive root: `/content/drive/MyDrive/perception_colab_data.zip`
  (contains `data/derived/bin/...`, `data/derived/labels_openpcdet/...`, and `checkpoints/pretrained/pointpillar_7728.pth`).

> Fine-tunes from KITTI `pointpillar_7728.pth`, 3-class head (Car/Pedestrian/Cyclist; our data has Car+Pedestrian). Range `[-40.96,-40.96,-0.5,40.96,40.96,3.0]`, voxel `[0.16,0.16,3.5]`, anchor bottom-heights re-measured for the ~0.3 m mount.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU! Runtime -> Change runtime type -> GPU, then re-run.')

## 2. Clone our repo (public, HTTPS)

In [ ]:
!git clone https://github.com/Abdulrahman2200925/perception.git /content/perception
%cd /content/perception
!git log --oneline -3

## 3. Mount Drive + unzip data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json
ZIP = '/content/drive/MyDrive/perception_colab_data.zip'
assert os.path.exists(ZIP), f'Data zip not found at {ZIP} - upload it to your Drive root.'

# unzip into the repo so relative paths line up (DATA_PATH=/content/perception)
!unzip -q -o "$ZIP" -d /content/perception

bins = glob.glob('/content/perception/data/derived/bin/*/*.bin')
man  = '/content/perception/data/derived/labels_openpcdet/MANIFEST_pairing.json'
ckpt = '/content/perception/checkpoints/pretrained/pointpillar_7728.pth'
print('bin files:', len(bins))
assert os.path.exists(man),  f'missing {man}'
assert os.path.exists(ckpt), f'missing {ckpt}'
assert len(bins) == 48, f'expected 48 .bin, found {len(bins)}'
print('manifest frames:', len(json.load(open(man))['frames']))
print('pretrained ckpt:', os.path.getsize(ckpt)//1024//1024, 'MB')
print('data OK')

## 4. Build OpenPCDet (pinned commit `846cf3e`)

⚠️ **spconv / CUDA must match the Colab runtime.** OpenPCDet needs `spconv` built for the runtime's CUDA. The cell picks the wheel from `torch.version.cuda` (e.g. `spconv-cu120` for CUDA 12.x, `spconv-cu118` for 11.8). If `import spconv` fails, fix that first (install the wheel matching `!nvcc --version`). Build errors are printed verbatim.

In [ ]:
%cd /content
!git clone https://github.com/open-mmlab/OpenPCDet.git /content/OpenPCDet
%cd /content/OpenPCDet
!git checkout 846cf3e
!git log --oneline -1

# spconv matching Colab CUDA
import torch
cu = (torch.version.cuda or '').replace('.', '')
spconv_pkg = 'spconv-cu%s' % (cu[:3] if cu else '120')
print('torch CUDA:', torch.version.cuda, '-> installing', spconv_pkg)
!pip install -q {spconv_pkg} || pip install -q spconv-cu120

# OpenPCDet deps + build
!pip install -q -r requirements.txt
!python setup.py develop

import pcdet
print('pcdet', pcdet.__version__)

## 5. Inject our dataset class + configs into OpenPCDet

In [ ]:
import os, shutil, re

# 5a. dataset class -> pcdet/datasets/mid360/
dst_dir = '/content/OpenPCDet/pcdet/datasets/mid360'
os.makedirs(dst_dir, exist_ok=True)
open(os.path.join(dst_dir, '__init__.py'), 'a').close()
shutil.copy('/content/perception/mine/dataset/mid360_dataset.py',
            os.path.join(dst_dir, 'mid360_dataset.py'))

# 5b. register Mid360Dataset in pcdet/datasets/__init__.py (idempotent)
init_p = '/content/OpenPCDet/pcdet/datasets/__init__.py'
src = open(init_p).read()
if 'Mid360Dataset' not in src:
    src = src.replace('from .dataset import DatasetTemplate',
                      'from .dataset import DatasetTemplate\nfrom .mid360.mid360_dataset import Mid360Dataset', 1)
    src = src.replace('__all__ = {',
                      "__all__ = {\n    'Mid360Dataset': Mid360Dataset,", 1)
    open(init_p, 'w').write(src)
    print('registered Mid360Dataset')
else:
    print('Mid360Dataset already registered')
assert 'Mid360Dataset' in open(init_p).read()

# 5c. copy configs into OpenPCDet tools/cfgs + fix Colab paths
os.makedirs('/content/OpenPCDet/tools/cfgs/mid360_models', exist_ok=True)
ds_dst = '/content/OpenPCDet/tools/cfgs/dataset_configs/mid360_dataset.yaml'
md_dst = '/content/OpenPCDet/tools/cfgs/mid360_models/mid360_pointpillar.yaml'
shutil.copy('/content/perception/configs/mid360_dataset.yaml', ds_dst)
shutil.copy('/content/perception/configs/mid360_pointpillar.yaml', md_dst)

# dataset cfg: DATA_PATH -> /content/perception
s = open(ds_dst).read()
s = re.sub(r"DATA_PATH:.*", "DATA_PATH: '/content/perception'", s, count=1)
open(ds_dst, 'w').write(s)

# model cfg: _BASE_CONFIG_ -> the copied dataset cfg (relative to tools/)
s = open(md_dst).read()
s = re.sub(r"_BASE_CONFIG_:.*", "_BASE_CONFIG_: cfgs/dataset_configs/mid360_dataset.yaml", s, count=1)
open(md_dst, 'w').write(s)

print('DATA_PATH    ->', [l for l in open(ds_dst) if l.strip().startswith('DATA_PATH')][0].strip())
print('_BASE_CONFIG_->', [l for l in open(md_dst) if '_BASE_CONFIG_' in l][0].strip())

## 6. Full smoke test (voxelization + collate)

In [ ]:
from pathlib import Path
import yaml
from easydict import EasyDict
from pcdet.utils import common_utils
from pcdet.datasets.mid360.mid360_dataset import Mid360Dataset

dcfg = EasyDict(yaml.safe_load(open('/content/OpenPCDet/tools/cfgs/dataset_configs/mid360_dataset.yaml')))
logger = common_utils.create_logger()
try:
    ds = Mid360Dataset(dataset_cfg=dcfg, class_names=['Car','Pedestrian','Cyclist'],
                       training=True, root_path=Path('/content/perception'), logger=logger)
    print('dataset len:', len(ds))
    d = ds[0]
    print('data_dict keys:', list(d.keys()))
    for k in ['points','voxels','voxel_coords','voxel_num_points','gt_boxes']:
        if k in d:
            print(f'  {k}: shape={getattr(d[k], "shape", None)}')
    print('frame_id:', d.get('frame_id'))
    assert 'voxels' in d, 'voxelization did not run'
    assert d['gt_boxes'].shape[1] == 8, f"gt_boxes should be (M,8), got {d['gt_boxes'].shape}"
    batch = ds.collate_batch([ds[0], ds[1]])
    print('collated batch keys:', list(batch.keys()), '| batch_size:', batch['batch_size'])
    print('SMOKE TEST PASS - full pipeline (load->augment->encode->voxelize->collate) OK')
except Exception as e:
    import traceback; traceback.print_exc()
    raise SystemExit('SMOKE TEST FAILED - fix before training. See error above.')

## 7. Overfit training (fine-tune, no eval)

Fine-tunes from `pointpillar_7728.pth` on all 48 frames. **Success = the training loss falling toward ~0** (overfit) - that proves the pipeline.

**On evaluation:** `Mid360Dataset` intentionally has no `evaluation()` (val skipped for this proof). OpenPCDet's `train.py` runs a post-training eval by default, which will raise on this dataset **after the checkpoint is already saved** - that error is **expected and ignorable** here; the loss curve is the signal, and Cell 8 copies the saved checkpoint regardless. (A clean no-op `evaluation()` is a later follow-up if silent runs are wanted.)

In [ ]:
%cd /content/OpenPCDet/tools
!python train.py \
  --cfg_file cfgs/mid360_models/mid360_pointpillar.yaml \
  --pretrained_model /content/perception/checkpoints/pretrained/pointpillar_7728.pth \
  --batch_size 2 --epochs 80 --workers 2

## 8. Save checkpoint to Drive (survives session wipe)

In [ ]:
import glob, os, shutil
out = sorted(glob.glob('/content/OpenPCDet/output/**/ckpt/*.pth', recursive=True))
print('checkpoints produced:', len(out))
for c in out[-3:]:
    print('  ', c)
assert out, 'no checkpoint found - did training run?'
dst = '/content/drive/MyDrive/perception_ckpts'
os.makedirs(dst, exist_ok=True)
latest = out[-1]
shutil.copy(latest, dst)
print('saved to Drive:', os.path.join(dst, os.path.basename(latest)))

## 9. Next steps

- **Good result:** the training loss decreases steadily toward ~0 over the 80 epochs - the data->label->model pipeline is proven end-to-end on the target sensor.
- **Not in this notebook (next phase):**
  - ONNX export of the fine-tuned model (NVIDIA `CUDA-PointPillars/tool/export_onnx.py`),
  - matching `params.h` edits (range / voxel / anchor bottom-heights must mirror this cfg),
  - TensorRT `.plan` rebuild + ROS2 deployment on the Jetson.
- After the overfit proof: collect/label more data, add a real train/val split + `evaluation()`, and re-tune anchors / LR.